¿Cuál usar?

Principiante: Keras/TensorFlow (más simple)
Producción/Investigación: PyTorch + TIMM (más flexible)
Estado del arte: TIMM (más funcionalidades)

Cada programa incluye ejemplos completos de:

✅ Carga del modelo preentrenado
✅ Transfer learning
✅ Entrenamiento y validación
✅ Predicción de imágenes
✅ Guardado/carga de modelos
✅ Data augmentation

1️⃣ TensorFlow/Keras (Oficial)

✅ Implementación oficial y más madura
Incluye: predicción, transfer learning, data augmentation, fine-tuning
Fácil de usar para principiantes
Mejor integración con TensorFlow Serving para producción

In [2]:
"""
Ejemplo de uso de InceptionResNetV2 con TensorFlow/Keras
Incluye: carga del modelo, predicción, fine-tuning y entrenamiento
"""

import tensorflow as tf
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import numpy as np

# ========================================
# 1. CARGAR MODELO PREENTRENADO (ImageNet)
# ========================================
print("Cargando InceptionResNetV2 preentrenado...")
model = InceptionResNetV2(weights='imagenet', include_top=True)
print(f"Modelo cargado. Total de parámetros: {model.count_params():,}")

# ========================================
# 2. PREDICCIÓN CON IMAGEN DE EJEMPLO
# ========================================
def predict_image(img_path, model):
    """Realiza predicción sobre una imagen"""
    img = image.load_img(img_path, target_size=(299, 299))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    
    preds = model.predict(x)
    decoded = decode_predictions(preds, top=3)[0]
    
    print("\nPredicciones:")
    for i, (imagenet_id, label, score) in enumerate(decoded):
        print(f"{i + 1}. {label}: {score*100:.2f}%")
    
    return decoded

# Ejemplo de predicción (necesitarías una imagen real)
# predict_image('elephant.jpg', model)

# ========================================
# 3. TRANSFER LEARNING - CLASIFICACIÓN PERSONALIZADA
# ========================================
def create_custom_model(num_classes=10, trainable_layers=50):
    """
    Crea modelo personalizado para transfer learning
    
    Args:
        num_classes: número de clases de salida
        trainable_layers: número de capas finales a entrenar
    """
    # Cargar modelo base sin la capa superior
    base_model = InceptionResNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(299, 299, 3)
    )
    
    # Congelar capas base inicialmente
    base_model.trainable = False
    
    # Añadir capas personalizadas
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    
    # Crear modelo completo
    custom_model = Model(inputs=base_model.input, outputs=predictions)
    
    print(f"\nModelo personalizado creado con {num_classes} clases")
    print(f"Capas congeladas: {len(base_model.layers)}")
    
    return custom_model, base_model

# Crear modelo personalizado
custom_model, base_model = create_custom_model(num_classes=10)

# ========================================
# 4. COMPILAR Y ENTRENAR
# ========================================
def compile_and_train(model, train_data, val_data, epochs_frozen=5, epochs_fine_tune=10):
    """
    Entrena el modelo en dos fases:
    1. Solo nuevas capas
    2. Fine-tuning de capas superiores
    """
    
    # FASE 1: Entrenar solo capas nuevas
    print("\n=== FASE 1: Entrenamiento de capas superiores ===")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history_1 = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs_frozen,
        verbose=1
    )
    
    # FASE 2: Fine-tuning
    print("\n=== FASE 2: Fine-tuning ===")
    # Descongelar las últimas capas del modelo base
    base_model.trainable = True
    for layer in base_model.layers[:-50]:
        layer.trainable = False
    
    # Recompilar con learning rate más bajo
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history_2 = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs_fine_tune,
        verbose=1
    )
    
    return history_1, history_2

# ========================================
# 5. EJEMPLO DE GENERADOR DE DATOS
# ========================================
def create_data_generators(train_dir, val_dir, batch_size=32):
    """Crea generadores de datos con data augmentation"""
    
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=preprocess_input,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.15,
        shear_range=0.15,
        fill_mode='nearest'
    )
    
    val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=preprocess_input
    )
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(299, 299),
        batch_size=batch_size,
        class_mode='categorical'
    )
    
    val_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=(299, 299),
        batch_size=batch_size,
        class_mode='categorical'
    )
    
    return train_generator, val_generator

# Ejemplo de uso (necesitarías directorios reales):
# train_gen, val_gen = create_data_generators('data/train', 'data/val')
# history_1, history_2 = compile_and_train(custom_model, train_gen, val_gen)

# ========================================
# 6. GUARDAR Y CARGAR MODELO
# ========================================
def save_model(model, filepath='inception_resnet_v2_custom.h5'):
    """Guarda el modelo completo"""
    model.save(filepath)
    print(f"\nModelo guardado en: {filepath}")

def load_saved_model(filepath='inception_resnet_v2_custom.h5'):
    """Carga un modelo guardado"""
    model = tf.keras.models.load_model(filepath)
    print(f"\nModelo cargado desde: {filepath}")
    return model

# ========================================
# 7. VISUALIZACIÓN DE ARQUITECTURA
# ========================================
def print_model_summary(model):
    """Imprime resumen del modelo"""
    print("\n" + "="*60)
    print("RESUMEN DEL MODELO")
    print("="*60)
    model.summary()
    
    print("\nCapas entrenables:")
    trainable_count = sum([1 for layer in model.layers if layer.trainable])
    print(f"Total: {trainable_count} de {len(model.layers)} capas")

# Ver resumen
print_model_summary(custom_model)

print("\n¡Ejemplo completo de InceptionResNetV2 con Keras/TensorFlow!")
print("\nPara usar este código:")
print("1. Instala: pip install tensorflow")
print("2. Prepara tus datos en directorios train/ y val/")
print("3. Descomentar las líneas de entrenamiento")

Cargando InceptionResNetV2 preentrenado...
225209952/225209952 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step 
Modelo cargado. Total de parámetros: 55,873,736

Modelo personalizado creado con 10 clases
Capas congeladas: 780

RESUMEN DEL MODELO


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_406 (Conv2D) │ (None, 149, 149,  │        864 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 149, 149,  │         96 │ conv2d_406[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_406      │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_407 (Conv2D) │ (None, 147, 147,  │      9,216 │ activation_406[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_407[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_407      │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_408 (Conv2D) │ (None, 147, 147,  │     18,432 │ activation_407[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_408[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_408      │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 73, 73,    │          0 │ activation_408[0… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_409 (Conv2D) │ (None, 73, 73,    │      5,120 │ max_pooling2d_8[… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_409[0][0]  │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_409      │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_410 (Conv2D) │ (None, 71, 71,    │    138,240 │ activation_409[0… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_410[0][0]  │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_410      │ (None, 71, 71,    │          0 │ batch_normalizat

 Total params: 56,440,554 (215.30 MB)

 Trainable params: 2,103,818 (8.03 MB)

 Non-trainable params: 54,336,736 (207.28 MB)


Capas entrenables:
Total: 6 de 786 capas

¡Ejemplo completo de InceptionResNetV2 con Keras/TensorFlow!

Para usar este código:
1. Instala: pip install tensorflow
2. Prepara tus datos en directorios train/ y val/
3. Descomentar las líneas de entrenamiento


2️⃣ PyTorch con TIMM (Recomendado)

✅ Más flexible y moderno
Usa TIMM (PyTorch Image Models) - la mejor librería para modelos de visión
Incluye: training completo, mixed precision, checkpoints
Ideal para investigación y desarrollo

In [ ]:
"""
Ejemplo de uso de InceptionResNetV2 con PyTorch
Nota: torchvision no incluye InceptionResNetV2, pero podemos usar timm o implementación custom
Aquí usamos una implementación basada en la estructura original
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

# ========================================
# 1. CARGAR MODELO DESDE TIMM (recomendado)
# ========================================
try:
    import timm
    
    def load_pretrained_model(num_classes=1000, pretrained=True):
        """
        Carga InceptionResNetV2 usando timm (PyTorch Image Models)
        Esta es la forma más sencilla en PyTorch
        """
        model = timm.create_model(
            'inception_resnet_v2',
            pretrained=pretrained,
            num_classes=num_classes
        )
        return model
    
    print("Cargando InceptionResNetV2 con timm...")
    model = load_pretrained_model(pretrained=True)
    print(f"Modelo cargado exitosamente")
    print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")
    
except ImportError:
    print("Timm no está instalado. Instalalo con: pip install timm")
    model = None

# ========================================
# 2. CONFIGURAR DISPOSITIVO (CPU/GPU)
# ========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsando dispositivo: {device}")

if model is not None:
    model = model.to(device)

# ========================================
# 3. TRANSFORMACIONES DE DATOS
# ========================================
def get_transforms():
    """Define las transformaciones para entrenamiento y validación"""
    
    train_transform = transforms.Compose([
        transforms.Resize(342),  # Resize más grande
        transforms.RandomCrop(299),  # Crop al tamaño requerido
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(342),
        transforms.CenterCrop(299),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    return train_transform, val_transform

# ========================================
# 4. CREAR DATALOADERS
# ========================================
def create_dataloaders(train_dir, val_dir, batch_size=32, num_workers=4):
    """Crea DataLoaders para entrenamiento y validación"""
    
    train_transform, val_transform = get_transforms()
    
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
    val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    print(f"\nDatasets cargados:")
    print(f"Train: {len(train_dataset)} imágenes")
    print(f"Val: {len(val_dataset)} imágenes")
    print(f"Clases: {len(train_dataset.classes)}")
    
    return train_loader, val_loader, train_dataset.classes

# ========================================
# 5. TRANSFER LEARNING - MODELO PERSONALIZADO
# ========================================
def create_custom_model(num_classes, freeze_backbone=True):
    """
    Crea modelo personalizado para transfer learning
    
    Args:
        num_classes: número de clases de salida
        freeze_backbone: si congelar capas del backbone
    """
    if model is None:
        raise Exception("Modelo no disponible. Instala timm primero.")
    
    # Cargar modelo base
    base_model = timm.create_model('inception_resnet_v2', pretrained=True)
    
    # Congelar capas si es necesario
    if freeze_backbone:
        for param in base_model.parameters():
            param.requires_grad = False
    
    # Obtener el número de features de entrada del clasificador
    num_features = base_model.get_classifier().in_features
    
    # Reemplazar el clasificador
    base_model.reset_classifier(num_classes)
    
    # O crear un clasificador personalizado más complejo:
    base_model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    
    return base_model

# Ejemplo: Crear modelo para 10 clases
if model is not None:
    custom_model = create_custom_model(num_classes=10, freeze_backbone=True)
    custom_model = custom_model.to(device)
    print("\nModelo personalizado creado")

# ========================================
# 6. FUNCIÓN DE ENTRENAMIENTO
# ========================================
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Entrena el modelo por una época"""
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0
    
    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Estadísticas
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total_samples += inputs.size(0)
    
    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects.double() / total_samples
    
    return epoch_loss, epoch_acc.item()

# ========================================
# 7. FUNCIÓN DE VALIDACIÓN
# ========================================
def validate(model, dataloader, criterion, device):
    """Valida el modelo"""
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)
    
    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects.double() / total_samples
    
    return epoch_loss, epoch_acc.item()

# ========================================
# 8. ENTRENAMIENTO COMPLETO
# ========================================
def train_model(model, train_loader, val_loader, num_epochs=10, lr=0.001):
    """
    Entrena el modelo completo
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    print("\nIniciando entrenamiento...")
    print("="*70)
    
    for epoch in range(num_epochs):
        print(f"\nÉpoca {epoch+1}/{num_epochs}")
        print("-" * 70)
        
        # Entrenar
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Validar
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # Actualizar scheduler
        scheduler.step(val_loss)
        
        # Guardar historial
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # Guardar mejor modelo
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_inception_resnet_v2.pth')
            print(f"✓ Mejor modelo guardado (acc: {best_acc:.4f})")
    
    print("\n" + "="*70)
    print(f"Entrenamiento completado. Mejor accuracy: {best_acc:.4f}")
    
    return history

# ========================================
# 9. PREDICCIÓN
# ========================================
def predict_image(model, image_path, class_names, device):
    """Realiza predicción sobre una imagen"""
    _, val_transform = get_transforms()
    
    from PIL import Image
    img = Image.open(image_path).convert('RGB')
    img_tensor = val_transform(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        top5_prob, top5_idx = torch.topk(probabilities, 5)
    
    print(f"\nPredicciones para {image_path}:")
    for i in range(5):
        class_name = class_names[top5_idx[0][i]]
        prob = top5_prob[0][i].item()
        print(f"{i+1}. {class_name}: {prob*100:.2f}%")

# ========================================
# 10. GUARDAR Y CARGAR MODELO
# ========================================
def save_checkpoint(model, optimizer, epoch, filepath='checkpoint.pth'):
    """Guarda checkpoint completo"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint guardado: {filepath}")

def load_checkpoint(model, optimizer, filepath='checkpoint.pth'):
    """Carga checkpoint"""
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    print(f"Checkpoint cargado: época {epoch}")
    return epoch

# ========================================
# EJEMPLO DE USO COMPLETO
# ========================================
if __name__ == "__main__":
    print("\n¡Ejemplo completo de InceptionResNetV2 con PyTorch!")
    print("\nPara usar este código:")
    print("1. Instala: pip install torch torchvision timm")
    print("2. Prepara tus datos en directorios train/ y val/")
    print("3. Ejecuta el entrenamiento")
    print("\nEjemplo:")
    print("  train_loader, val_loader, classes = create_dataloaders('data/train', 'data/val')")
    print("  history = train_model(custom_model, train_loader, val_loader, num_epochs=10)")

3️⃣ TIMM Puro (Más Avanzado)

✅ Implementación más completa y profesional
700+ modelos preentrenados disponibles
Optimizadores avanzados (AdamW, LAMB, etc.)
Mixed precision automático
Exportación a ONNX
Transformaciones automáticas basadas en el modelo

In [ ]:
"""
Ejemplo completo de InceptionResNetV2 con TIMM (PyTorch Image Models)
TIMM es la librería más versátil y actualizada para modelos de visión en PyTorch
"""

import torch
import torch.nn as nn
import timm
from timm.data import resolve_data_config, create_transform
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from timm.optim import create_optimizer_v2
from timm.scheduler import create_scheduler
from torch.utils.data import DataLoader
from torchvision import datasets
from PIL import Image
import json

# ========================================
# 1. EXPLORAR MODELOS DISPONIBLES
# ========================================
def list_inception_models():
    """Lista todos los modelos Inception disponibles en timm"""
    inception_models = timm.list_models('*inception*', pretrained=True)
    print("Modelos Inception disponibles en timm:")
    print("="*60)
    for i, model_name in enumerate(inception_models, 1):
        print(f"{i}. {model_name}")
    print("="*60)
    return inception_models

# Ver modelos disponibles
available_models = list_inception_models()

# ========================================
# 2. CARGAR MODELO PREENTRENADO
# ========================================
def load_model(model_name='inception_resnet_v2', pretrained=True, num_classes=1000):
    """
    Carga modelo InceptionResNetV2
    
    Args:
        model_name: nombre del modelo en timm
        pretrained: cargar pesos preentrenados
        num_classes: número de clases (1000 para ImageNet)
    """
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=num_classes
    )
    
    # Información del modelo
    print(f"\n{'='*60}")
    print(f"Modelo: {model_name}")
    print(f"Pretrained: {pretrained}")
    print(f"Clases: {num_classes}")
    print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"{'='*60}")
    
    return model

# Cargar modelo base
model = load_model('inception_resnet_v2', pretrained=True)

# ========================================
# 3. CONFIGURACIÓN DEL MODELO
# ========================================
def get_model_info(model):
    """Obtiene información detallada del modelo"""
    config = model.default_cfg
    print("\nConfiguración del modelo:")
    print(f"  Input size: {config['input_size']}")
    print(f"  Mean: {config['mean']}")
    print(f"  Std: {config['std']}")
    print(f"  Interpolation: {config['interpolation']}")
    print(f"  Crop pct: {config['crop_pct']}")
    return config

config = get_model_info(model)

# ========================================
# 4. TRANSFORMACIONES AUTOMÁTICAS
# ========================================
def get_transforms_timm(model, is_training=False):
    """
    Crea transformaciones usando la configuración del modelo
    TIMM hace esto automáticamente basándose en el modelo
    """
    config = resolve_data_config({}, model=model)
    transform = create_transform(**config, is_training=is_training)
    
    print(f"\nTransformación {'entrenamiento' if is_training else 'validación'}:")
    print(transform)
    
    return transform

train_transform = get_transforms_timm(model, is_training=True)
val_transform = get_transforms_timm(model, is_training=False)

# ========================================
# 5. CREAR MODELO PARA TRANSFER LEARNING
# ========================================
def create_transfer_model(model_name='inception_resnet_v2', 
                         num_classes=10, 
                         freeze_backbone=True,
                         drop_rate=0.2):
    """
    Crea modelo para transfer learning con opciones avanzadas
    
    Args:
        model_name: nombre del modelo base
        num_classes: número de clases objetivo
        freeze_backbone: congelar backbone
        drop_rate: tasa de dropout
    """
    # Cargar modelo con nuevo número de clases
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes,
        drop_rate=drop_rate,  # Dropout en el clasificador
        drop_path_rate=0.1    # Stochastic depth
    )
    
    # Congelar backbone si es necesario
    if freeze_backbone:
        # Congelar todas las capas excepto el clasificador
        for name, param in model.named_parameters():
            if 'classif' not in name:
                param.requires_grad = False
    
    # Ver qué capas son entrenables
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    
    print(f"\nTransfer Learning Model:")
    print(f"  Total parámetros: {total_params:,}")
    print(f"  Parámetros entrenables: {trainable_params:,}")
    print(f"  Porcentaje entrenable: {100*trainable_params/total_params:.2f}%")
    
    return model

# Crear modelo personalizado
custom_model = create_transfer_model(
    model_name='inception_resnet_v2',
    num_classes=10,
    freeze_backbone=True,
    drop_rate=0.3
)

# ========================================
# 6. DESCONGELAR CAPAS PROGRESIVAMENTE
# ========================================
def unfreeze_layers(model, num_layers_to_unfreeze=None):
    """
    Descongela capas del modelo progresivamente
    
    Args:
        num_layers_to_unfreeze: número de capas a descongelar desde el final
                                None = descongelar todas
    """
    layers = list(model.named_parameters())
    
    if num_layers_to_unfreeze is None:
        # Descongelar todo
        for name, param in layers:
            param.requires_grad = True
        print(f"Todas las capas descongeladas")
    else:
        # Descongelar últimas N capas
        for name, param in layers[-num_layers_to_unfreeze:]:
            param.requires_grad = True
        print(f"Descongeladas últimas {num_layers_to_unfreeze} capas")
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parámetros entrenables: {trainable:,}")

# ========================================
# 7. CREAR DATALOADERS CON TIMM
# ========================================
def create_dataloaders_timm(train_dir, val_dir, model, batch_size=32, workers=4):
    """Crea dataloaders con transformaciones de timm"""
    
    train_transform = get_transforms_timm(model, is_training=True)
    val_transform = get_transforms_timm(model, is_training=False)
    
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
    val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=workers,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=True
    )
    
    return train_loader, val_loader, train_dataset.classes

# ========================================
# 8. OPTIMIZADOR Y SCHEDULER DE TIMM
# ========================================
def create_optimizer_and_scheduler(model, epochs, train_loader):
    """
    Crea optimizador y scheduler usando las funciones de timm
    Timm incluye muchos optimizadores avanzados
    """
    # Optimizador (timm soporta: sgd, adam, adamw, rmsprop, etc.)
    optimizer = create_optimizer_v2(
        model,
        opt='adamw',           # Optimizer type
        lr=0.001,              # Learning rate
        weight_decay=0.05      # Weight decay
    )
    
    # Scheduler (timm soporta muchos tipos)
    scheduler, num_epochs = create_scheduler(
        args=type('Args', (), {
            'sched': 'cosine',           # cosine, step, plateau, etc.
            'epochs': epochs,
            'decay_rate': 0.1,
            'warmup_lr': 1e-6,
            'warmup_epochs': 3,
            'cooldown_epochs': 5,
            'min_lr': 1e-6
        })(),
        optimizer=optimizer
    )
    
    print(f"\nOptimizer: {type(optimizer).__name__}")
    print(f"Scheduler: {type(scheduler).__name__}")
    
    return optimizer, scheduler

# ========================================
# 9. FUNCIONES DE PÉRDIDA AVANZADAS
# ========================================
def get_loss_function(loss_type='ce', smoothing=0.1):
    """
    Obtiene función de pérdida
    
    loss_type:
        'ce': CrossEntropy normal
        'ls': Label Smoothing
        'st': Soft Target (para distilación)
    """
    if loss_type == 'ls':
        criterion = LabelSmoothingCrossEntropy(smoothing=smoothing)
        print(f"Loss: Label Smoothing CE (smoothing={smoothing})")
    elif loss_type == 'st':
        criterion = SoftTargetCrossEntropy()
        print("Loss: Soft Target CE")
    else:
        criterion = nn.CrossEntropyLoss()
        print("Loss: Standard CrossEntropy")
    
    return criterion

# ========================================
# 10. PREDICCIÓN CON MODELO
# ========================================
def predict_image(model, image_path, top_k=5):
    """
    Realiza predicción sobre una imagen
    Usa las transformaciones correctas automáticamente
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    # Cargar y transformar imagen
    img = Image.open(image_path).convert('RGB')
    transform = get_transforms_timm(model, is_training=False)
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # Predicción
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
    
    # Top-K predicciones
    top_prob, top_idx = torch.topk(probabilities, top_k)
    
    print(f"\nPredicciones para {image_path}:")
    print("="*50)
    for i in range(top_k):
        idx = top_idx[0][i].item()
        prob = top_prob[0][i].item()
        print(f"{i+1}. Clase {idx}: {prob*100:.2f}%")
    print("="*50)
    
    return top_idx[0].cpu().numpy(), top_prob[0].cpu().numpy()

# ========================================
# 11. ENTRENAMIENTO CON MIXED PRECISION
# ========================================
def train_with_amp(model, train_loader, val_loader, epochs=10):
    """
    Entrena con Automatic Mixed Precision para mayor velocidad
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # Optimizer y scheduler
    optimizer, scheduler = create_optimizer_and_scheduler(
        model, epochs, train_loader
    )
    
    # Loss
    criterion = get_loss_function('ls', smoothing=0.1)
    
    # Scaler para AMP
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
    
    best_acc = 0.0
    
    print("\n" + "="*70)
    print("Iniciando entrenamiento con Mixed Precision")
    print("="*70)
    
    for epoch in range(epochs):
        # Entrenamiento
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            # Mixed precision forward pass
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
        
        # Validación
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        # Métricas
        train_acc = 100. * train_correct / train_total
        val_acc = 100. * val_correct / val_total
        
        print(f"\nÉpoca {epoch+1}/{epochs}")
        print(f"Train Loss: {train_loss/len(train_loader):.4f} | Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss/len(val_loader):.4f} | Acc: {val_acc:.2f}%")
        
        # Guardar mejor modelo
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_inception_resnet_timm.pth')
            print(f"✓ Mejor modelo guardado (acc: {best_acc:.2f}%)")
        
        # Step scheduler
        if scheduler is not None:
            scheduler.step(epoch)
    
    print(f"\nMejor accuracy alcanzado: {best_acc:.2f}%")

# ========================================
# 12. EXPORTAR MODELO A ONNX
# ========================================
def export_to_onnx(model, output_path='inception_resnet_v2.onnx'):
    """Exporta el modelo a formato ONNX"""
    device = torch.device('cpu')
    model = model.to(device)
    model.eval()
    
    # Input dummy
    dummy_input = torch.randn(1, 3, 299, 299).to(device)
    
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    
    print(f"\nModelo exportado a ONNX: {output_path}")

# ========================================
# INFORMACIÓN Y USO
# ========================================
if __name__ == "__main__":
    print("\n" + "="*70)
    print("EJEMPLO COMPLETO DE INCEPTIONRESNETV2 CON TIMM")
    print("="*70)
    print("\n📦 Instalación:")
    print("   pip install timm torch torchvision pillow")
    print("\n🚀 Ventajas de TIMM:")
    print("   • 700+ modelos preentrenados")
    print("   • Transformaciones automáticas")
    print("   • Optimizadores y schedulers avanzados")
    print("   • Mixed precision nativo")
    print("   • Documentación excelente")
    print("\n💡 Uso básico:")
    print("   model = timm.create_model('inception_resnet_v2', pretrained=True)")
    print("   # ¡Y listo para usar!")
    print("="*70)